# Datamine EXTEND Process Example

This notebook demonstrates how to configure and run the **`extend`** process wrapper in `dmstudio`.

## Process Description

## EXTEND

Extends an existing file with data entered in free format. The method of supplying the new data is described in full in the instructions for process [INDATA](<indata.md>).

Records may not start with the character !. This is because the ! symbol acts as the end-of-data character. However, macro files, where the command starts with !, may be read if a blank is inserted prior to the ! symbol; but because records in a macro file are terminated as soon as the first comma is met, EXTEND is not suitable for extending macro files; [EXTNDF](<extndf.md>) is preferable.

### About SYSFILE

SYSFILE is an external system file name from which the free format data is to be read, or return for entry from the keyboard. Format of the system file name is system specific, with a maximum of 56 characters allowed, including pathnames.

### About DATA

If no SYSFILE exists, data lines are entered in free format. ! terminates. The rules for free format data entry are:

1. Data line must not be more than 80 characters in length.

2. Text or numeric data for each field must be separated by commas. A comma is therefore not a legal character in a text field.

3. Text fields are automatically filled with blanks to their full length, if the string given is shorter than that required. Text fields which are too long are truncated.

4. Text fields are not entered in quotes (').

5. An absent data code is inserted for any field coded just by a comma.

6. Repetition of the previous value of a field in the previous record is achieved by use of the ditto mark ("). This mark cannot therefore be used within text fields.

### Input Files:

* **in** (Undefined):
  File to be extended.
  Required=Yes

### Output Files:

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('extend')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute extend
print("Running extend...")
dm_cmd.extend(
    in_i='t_assays',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("extend execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_extend_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")